# ROS Tutorial: Making Robots from Scratch

Welcome to the bridge between Software Engineering and physical Hardware. While Machine Learning (ML) and Artificial Intelligence (AI) provide the "brain" of a robot, they need a nervous system to communicate with sensors (eyes/ears) and actuators (muscles/wheels). 

The **Robot Operating System (ROS)**, specifically modern ROS 2, is the industry standard for this. It is not a traditional OS like Windows or Linux; rather, it is a robust middleware framework. 

### The Analytical Architecture of ROS
ROS operates on a **Distributed Graph Architecture**:
1. **Nodes:** Independent processes. (e.g., A Python script running a Neural Network is one node; a C++ script reading a camera is another).
2. **Topics:** The data buses connecting nodes.
3. **Publishers & Subscribers:** Nodes communicate asynchronously. A camera node *publishes* image data to an "Image Topic". An ML node *subscribes* to that topic, processes the image, and *publishes* a steering command to a "Velocity Topic".

In this tutorial, we will analytically build a complete ROS 2 pipeline using Python (`rclpy`). We will create a simulated Lidar sensor (Publisher) and an AI control unit (Subscriber) that makes decisions based on that sensor data.

In [ ]:
# Install instructions (In a real environment, you must install ROS 2 on Ubuntu)
# !pip install -q rclpy 

import rclpy
from rclpy.node import Node

# ROS communicates using strictly typed messages to ensure memory safety and speed across languages
from std_msgs.msg import Float32, String
import random
import time

print("ROS 2 Python Library (rclpy) imported. Ready to build the robotic nervous system.")

## Step 1: Building the Sensor (The Publisher Node)

First, we need to gather data from the physical world. In robotics, sensors operate at specific frequencies (e.g., a LIDAR spinning at 10Hz). 

We will create a ROS Node that simulates a proximity sensor. Analytically, a ROS Publisher does not care *who* is listening. It simply blasts data into the ether on a specific Topic. This **decouples** the hardware from the software; your ML model doesn't need to know how the LIDAR works, it just needs the data.

In [ ]:
class ProximitySensorNode(Node):
    """
    Simulates a hardware sensor measuring distance to an obstacle.
    Inherits from the core rclpy Node class.
    """
    def __init__(self):
        # 1. Initialize the node with a unique name in the ROS Graph
        super().__init__('proximity_sensor')
        
        # 2. Create the Publisher
        # Arguments: Message Type (Float32), Topic Name ('obstacle_distance'), Queue Size (10)
        self.publisher_ = self.create_publisher(Float32, 'obstacle_distance', 10)
        
        # 3. Create a Timer to simulate the physical sensor's refresh rate (e.g., 2Hz / 0.5 seconds)
        timer_period = 0.5  
        self.timer = self.create_timer(timer_period, self.timer_callback)
        
        self.get_logger().info("Proximity Sensor Node has been started.")

    def timer_callback(self):
        """This function executes every 0.5 seconds to publish new data."""
        msg = Float32()
        
        # Simulate an object getting closer and further away (between 0.1 and 5.0 meters)
        msg.data = random.uniform(0.1, 5.0)
        
        # Publish the message to the ROS Graph
        self.publisher_.publish(msg)
        self.get_logger().info(f'Publishing Sensor Data: "{msg.data:.2f} meters"')

## Step 2: Building the "Brain" (The Subscriber Node)

Now that data is flowing through the ROS network on the `obstacle_distance` topic, we need an entity to ingest and process it. 

This is where your Artificial Intelligence lives. We will build a Subscriber Node that listens to the sensor. When a new message arrives, it triggers a **Callback Function**. Inside this callback, we will implement a simple analytical control logic: if the obstacle is closer than 1.0 meter, trigger an emergency stop.

In [ ]:
class AIBrainNode(Node):
    """
    Simulates the central processing unit or ML model of the robot.
    Listens to sensors and makes decisions.
    """
    def __init__(self):
        super().__init__('ai_brain')
        
        # 1. Create the Subscriber
        # Arguments: Message Type, Topic Name to listen to, Callback Function, Queue Size
        self.subscription = self.create_subscription(
            Float32,
            'obstacle_distance',
            self.sensor_callback,
            10
        )
        
        # Prevent unused variable warning
        self.subscription  
        
        self.get_logger().info("AI Brain Node is online and listening...")

    def sensor_callback(self, msg):
        """
        This function is automatically triggered WHENEVER a new message 
        appears on the 'obstacle_distance' topic.
        """
        distance = msg.data
        
        # Analytical Control Logic (In a real robot, this could be a Neural Network inference)
        if distance < 1.0:
            self.get_logger().warn(f'DANGER! Obstacle at {distance:.2f}m. Initiating EMERGENCY STOP.')
            # Here you would publish to a '/cmd_vel' topic to stop the motors
        else:
            self.get_logger().info(f'Path clear. Obstacle at {distance:.2f}m. Proceeding forward.')

## Step 3: Executing the ROS Graph (The Event Loop)

In standard Python scripts, code runs top-to-bottom and exits. ROS nodes, however, must run continuously to listen for messages and publish data. This requires an **Event Loop**.

In ROS 2, we use `rclpy.spin()`. Because we have two nodes (Sensor and Brain) and we want them to run simultaneously in this single notebook cell, we will use a `SingleThreadedExecutor`. 

*Note: In a production robotic environment, these nodes would be saved as separate `.py` files and launched via the terminal (e.g., `ros2 run my_package ai_brain`), utilizing the true distributed power of the OS.*

In [ ]:
from rclpy.executors import SingleThreadedExecutor

def run_robot_pipeline():
    """
    Initializes the ROS network, instantiates the nodes, and spins them.
    """
    # 1. Initialize the ROS 2 communications backbone
    rclpy.init()
    
    # 2. Instantiate our mathematical/hardware representations
    sensor_node = ProximitySensorNode()
    brain_node = AIBrainNode()
    
    # 3. Create an executor to manage both nodes concurrently
    executor = SingleThreadedExecutor()
    executor.add_node(sensor_node)
    executor.add_node(brain_node)
    
    print("--- ROS 2 Graph Execution Started ---")
    
    try:
        # Spin the executor. We use a small timeout to simulate just 5 cycles 
        # so this notebook cell doesn't run infinitely.
        for _ in range(5):
            executor.spin_once(timeout_sec=1.0)
            
    except KeyboardInterrupt:
        pass
    finally:
        # 4. Clean up the nodes and shutdown the ROS network gracefully
        sensor_node.destroy_node()
        brain_node.destroy_node()
        rclpy.shutdown()
        print("--- ROS 2 Graph Safely Terminated ---")

# Execute the simulation
run_robot_pipeline()

## Conclusion

You have successfully built the fundamental nervous system of a robot!

**Key Analytical Takeaways for ML Engineers:**
1. **Total Decoupling:** Notice how the `AIBrainNode` had zero knowledge of how `ProximitySensorNode` generated its data. As an ML engineer, this means you can train a model using a simulated sensor in Unity or Gazebo, and deploy it to a physical robot without changing a single line of your AI code, as long as the ROS Topic names match.
2. **Asynchronous Computation:** Your ML inference does not block the camera from capturing the next frame. The ROS queue handles the data stream, ensuring the system remains non-blocking and real-time.
3. **Scalability:** To upgrade this, you wouldn't rewrite the brain. You would simply add a new node (like a `CameraNode`), publish a `SensorImage` message, and add a second subscriber to your Brain to fuse the Lidar and Camera data together using multi-modal Machine Learning. 

With ROS handling the plumbing, you are entirely free to focus on the intelligence of the machine.